## Prepare the dataset for tokenization and training

In [ ]:
#Imports
!pip install transformers
!pip install pandas
!pip install transformers[torch]
!pip install accelerate -U
!pip install torch
!pip install tqdm
import pandas as pd
from transformers import logging
logging.set_verbosity_error()
import torch
from tqdm.notebook import tqdm

from transformers import BertTokenizer
from torch.utils.data import TensorDataset

from transformers import BertForSequenceClassification
import numpy as np



In [ ]:
df = pd.read_csv("./darknet-categories.csv")
df.head()

In [ ]:
df = df.loc[:, ['content', 'category']]
df = df.rename(columns={'content': 'Content', 'category': 'Category'})
df.head()

In [ ]:
df['Category'].value_counts()

Remove the values whose count is less than 50

In [ ]:
removed_categories = df['Category'].value_counts()
categories_less_than_50 = removed_categories[removed_categories < 50].index.tolist()
print(categories_less_than_50)
df_filtered = df[~df['Category'].isin(categories_less_than_50)]

Get the labels

In [ ]:
possible_labels = df_filtered.Category.unique()

label_dict = {}
for index, possible_label in enumerate(possible_labels):
    label_dict[possible_label] = index
label_dict

Balance the dataset examples

In [ ]:
df_csam = df[df['Category']=='Pedophilia']
df_drugs = df[df['Category']=='Drugs']
df_services = df[df['Category']=='Services']
df_fraud = df[df['Category']=='Fraud']
df_drugs_downsampled = df_drugs.sample(df_csam.shape[0]+100)
df_balanced = pd.concat([df_drugs_downsampled, df_csam, df_services,df_fraud])
df_balanced['Category'].value_counts()

In [ ]:
df_balanced['Category'].value_counts()

In [ ]:
df_balanced['label'] = df_balanced.Category.replace(label_dict)


Train and Validation set

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(df_balanced.index.values,
                                                  df_balanced.label.values,
                                                  test_size=0.15,
                                                  random_state=42,
                                                  stratify=df_balanced.label.values)

df_balanced['data_type'] = ['not_set']*df_balanced.shape[0]

df_balanced.loc[X_train, 'data_type'] = 'train'
df_balanced.loc[X_val, 'data_type'] = 'val'

df_balanced.groupby(['Category', 'label', 'data_type']).count()



BertTokeniker and encode data

In [ ]:
df = df_balanced
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased',
                                          do_lower_case=True)

encoded_data_train = tokenizer.batch_encode_plus(
    df[df.data_type=='train'].Content.values,
    add_special_tokens=True,
    return_attention_mask=True,
    pad_to_max_length=True,
    max_length=512,
    return_tensors='pt'
)

encoded_data_val = tokenizer.batch_encode_plus(
    df[df.data_type=='val'].Content.values,
    add_special_tokens=True,
    return_attention_mask=True,
    pad_to_max_length=True,
    max_length=512, #512 for BERT
    return_tensors='pt'
)


input_ids_train = encoded_data_train['input_ids']
attention_masks_train = encoded_data_train['attention_mask']
labels_train = torch.tensor(df[df.data_type=='train'].label.values)

input_ids_val = encoded_data_val['input_ids']
attention_masks_val = encoded_data_val['attention_mask']
labels_val = torch.tensor(df[df.data_type=='val'].label.values)

dataset_train = TensorDataset(input_ids_train, attention_masks_train, labels_train)
dataset_val = TensorDataset(input_ids_val, attention_masks_val, labels_val)

Pretrained model

In [ ]:

model = BertForSequenceClassification.from_pretrained("bert-base-uncased",
                                                      num_labels=len(label_dict),
                                                      output_attentions=False,
                                                      output_hidden_states=False)

DataLoaders

In [ ]:
from torch.utils.data import DataLoader, RandomSampler, SequentialSampler

batch_size = 3

dataloader_train = DataLoader(dataset_train,
                              sampler=RandomSampler(dataset_train),
                              batch_size=batch_size)

dataloader_validation = DataLoader(dataset_val,
                                   sampler=SequentialSampler(dataset_val),
                                   batch_size=batch_size)

Optimizer & Scheduler

In [ ]:
from transformers import AdamW, get_linear_schedule_with_warmup

optimizer = AdamW(model.parameters(),
                  lr=1e-5,
                  eps=1e-8)

epochs = 5

scheduler = get_linear_schedule_with_warmup(optimizer,
                                            num_warmup_steps=0,
                                            num_training_steps=len(dataloader_train)*epochs)

Performance Metrics

In [ ]:
from sklearn.metrics import f1_score

def f1_score_func(preds, labels):
    preds_flat = np.argmax(preds, axis=1).flatten()
    labels_flat = labels.flatten()
    return f1_score(labels_flat, preds_flat, average='weighted')

def accuracy_per_class(preds, labels):
    label_dict_inverse = {v: k for k, v in label_dict.items()}

    preds_flat = np.argmax(preds, axis=1).flatten()
    labels_flat = labels.flatten()

    for label in np.unique(labels_flat):
        y_preds = preds_flat[labels_flat==label]
        y_true = labels_flat[labels_flat==label]
        percentage = 100*len(y_preds[y_preds==label])/len(y_true)
        print(f'Class: {label_dict_inverse[label]}')
        print(f'Accuracy: {len(y_preds[y_preds==label])}/{len(y_true)} {percentage}%\n')

Mount the saved directory

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!mkdir -p "/content/drive/My Drive/models"
with open('/content/drive/MyDrive/models/test.txt','w') as file:
    file.write('This is a test')

Training Loop


In [ ]:
import random

seed_val = 17
save_dir = "/content/drive/My Drive/models"
random.seed(seed_val)
np.random.seed(seed_val)
torch.manual_seed(seed_val)
torch.cuda.manual_seed_all(seed_val)
device = torch.device('cpu')

def evaluate(dataloader_val):

    model.eval()

    loss_val_total = 0
    predictions, true_vals = [], []

    for batch in dataloader_val:

        batch = tuple(b.to(device) for b in batch)

        inputs = {'input_ids':      batch[0],
                  'attention_mask': batch[1],
                  'labels':         batch[2],
                 }

        with torch.no_grad():
            outputs = model(**inputs)

        loss = outputs[0]
        logits = outputs[1]
        loss_val_total += loss.item()

        logits = logits.detach().cpu().numpy()
        label_ids = inputs['labels'].cpu().numpy()
        predictions.append(logits)
        true_vals.append(label_ids)

    loss_val_avg = loss_val_total/len(dataloader_val)

    predictions = np.concatenate(predictions, axis=0)
    true_vals = np.concatenate(true_vals, axis=0)

    return loss_val_avg, predictions, true_vals

for epoch in tqdm(range(1, epochs+1)):

    model.train()

    loss_train_total = 0

    progress_bar = tqdm(dataloader_train, desc='Epoch {:1d}'.format(epoch), leave=False, disable=False)
    for batch in progress_bar:

        model.zero_grad()

        batch = tuple(b.to(device) for b in batch)

        inputs = {'input_ids':      batch[0],
                  'attention_mask': batch[1],
                  'labels':         batch[2],
                 }

        outputs = model(**inputs)

        loss = outputs[0]
        loss_train_total += loss.item()
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()
        scheduler.step()

        progress_bar.set_postfix({'training_loss': '{:.3f}'.format(loss.item()/len(batch))})

    try:
      torch.save(model.state_dict(), f'{save_dir}/finetuned_BERT_epoch_{epoch}.model')
    except:
      print("Could not save file")

    tqdm.write(f'\nEpoch {epoch}')

    loss_train_avg = loss_train_total/len(dataloader_train)
    tqdm.write(f'Training loss: {loss_train_avg}')

    val_loss, predictions, true_vals = evaluate(dataloader_validation)
    val_f1 = f1_score_func(predictions, true_vals)
    tqdm.write(f'Validation loss: {val_loss}')
    tqdm.write(f'F1 Score (Weighted): {val_f1}')

Evaluating the model

In [ ]:
from transformers import BertForSequenceClassification
import torch

model = BertForSequenceClassification.from_pretrained("bert-base-uncased",
                                                      num_labels=len(label_dict),
                                                      output_attentions=False,
                                                      output_hidden_states=False)

model.to(device)

model.load_state_dict(torch.load('/content/drive/My Drive/models/finetuned_BERT_epoch_1.model', map_location=torch.device('cpu')))

_, predictions, true_vals = evaluate(dataloader_validation)
accuracy_per_class(predictions, true_vals)

In [ ]:
accuracy_per_class(predictions, true_vals)

In [ ]:
print(dataloader_validation)

In [ ]:
import torch
from transformers import BertForSequenceClassification, BertTokenizer

# Load the pretrained BERT model and tokenizer (assuming you used BERT for training)
model_name = 'bert-base-uncased'  # Change this to match the architecture you used during training
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForSequenceClassification.from_pretrained(model_name,num_labels=4,output_attentions=False,
                                                      output_hidden_states=False)

# Load the fine-tuned model checkpoint (replace 'epoch_x' with the desired epoch number)
model_checkpoint = "/content/drive/My Drive/models/finetuned_BERT_epoch_1.model"
model.load_state_dict(torch.load(model_checkpoint))

# Put the model in evaluation mode
model.eval()

# Prepare input data for inference
text = open('./test.txt','r').read()
input_ids = tokenizer.encode(text, add_special_tokens=True, max_length=128, truncation=True, padding='max_length', return_tensors='pt')
attention_mask = input_ids != tokenizer.pad_token_id  # Create attention mask

# Perform inference
with torch.no_grad():
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)

# Get the predicted class probabilities (logits)
logits = outputs.logits

# Convert logits to probabilities using softmax
probabilities = torch.softmax(logits, dim=1)

# Get the predicted class label (index with highest probability)
predicted_class = torch.argmax(probabilities, dim=1).item()
label_resolver = {0 : 'Drugs', 1: 'Fraud', 2: 'Services', 3 : 'Pedophilia'}
predicted_label = label_resolver[predicted_class]

# Print the results
print("Predicted class probabilities:", probabilities)
print("Predicted class:", predicted_class)
print("Predicted label:", predicted_label)  # Uncomment if you have a label mapping


In [ ]:
label_dict